# CDC PLACES 2024 - Chronic Health Risk Modeling Project
## Phase 1 & Phase 2 Integrated Machine Learning Pipeline

This project analyzes the CDC PLACES 2024 Census-Tract Dataset, which provides model-based estimates for chronic diseases, preventive-care behaviors, mental and physical health indicators, and social determinants of health across approximately 73,000 U.S. census tracts.

The goal of this project is to build a two-phase machine learning pipeline that:

- Predicts chronic disease prevalence using behavioral, preventive-care, and social determinant predictors

- Identifies the strongest predictors for each chronic disease

- Constructs a Chronic Health Risk Index (CHRI) using a neural-network-based weighting approach

- Classifies census tracts into Low, Medium, and High risk tiers

- Generates tract-level and county-level health-risk visualizations

- Supports public health decision-making with interpretable ML results

# Research Questions
### RQ1 : Predictive Strength

How strongly do behavioral, preventive-care, and social-determinant predictors explain variation in chronic disease prevalence across U.S. census tracts?

### RQ2 : Feature Importance

Which predictors most influence each chronic condition (e.g., diabetes, obesity, high blood pressure, coronary heart disease, stroke)?

### RQ3 : Composite Health Burden

How can multiple chronic disease outcomes be combined into a single Chronic Health Risk Index (CHRI) that summarizes community-level disease burden?

### RQ4 : Health Risk Classification

Can census tracts be reliably classified into Low, Medium, and High risk levels using the CHRI?



# Two-Phase Machine Learning Framework
## Phase 1 — Regression Modeling

- 10 chronic disease outcomes used as target variables

- 25 behavioral, preventive-care, mental-health, and social determinant predictors

- Models evaluated: Random Forest and Gradient Boosting

- Extraction of top predictor variables for each disease using feature importance from Random Forest

## Phase 2 — Chronic Health Risk Index (CHRI)

- Neural network used to learn disease-level weights for CHRI construction

- CHRI scaled to a 0–100 population-health risk score

- Census tracts classified into Low / Medium / High risk tiers using quantile segmentation

- County-level CHRI calculated using population-weighted aggregation

In [ ]:
# STEP 1 — IMPORT LIBRARIES

# Core data libraries
import pandas as pd        # DataFrame handling
import numpy as np         # Numerical operations

# Machine learning (Phase 1 models and preprocessing)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Neural networks (Phase 2 CHRI modeling)
import torch
import torch.nn as nn
import torch.optim as optim

# Mapping for geospatial visualizations
import folium
from folium.plugins import MarkerCluster


This cell imports all core libraries required for
- Data processing
- Machine learning
- Neural-network training
- Geospatial visualization.
  
It supports the entire two-phase pipeline, including
- Data cleaning
- Regression modeling
- Neural-network-based CHRI construction
- Mapping.

In [ ]:
# STEP 2 — LOAD RAW DATASET

# Load raw CDC PLACES dataset (Census Tract level)
df_original = pd.read_csv("PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2024_release_20251010 (1).csv")

# Standardize column names to lowercase for consistency
df_original.columns = df_original.columns.str.lower()

# Fix population column: convert "1,370" → 1370
df_original['totalpop18plus'] = (
    df_original['totalpop18plus']
    .astype(str)
    .str.replace(",", "")
    .astype(float)
)

# ---output ---
print("Dataset loaded successfully.")
print("Shape:", df_original.shape)

print("\nColumn Names:")
print(df_original.columns.tolist())

# Display first 5 rows
df_original.head()


- This cell loads the CDC PLACES 2024 Census Tract dataset, standardizes column names, and fixes formatting issues.
Specifically, it converts population columns containing commas into proper numeric values and prepares the dataset for preprocessing in later steps.

In [ ]:
# STEP 3 — DEFINE TARGETS & PREDICTORS

# Target columns: 10 chronic diseases
target_cols = [
    'diabetes_crudeprev','obesity_crudeprev','bphigh_crudeprev',
    'bpmed_crudeprev','chd_crudeprev','stroke_crudeprev',
    'copd_crudeprev','cancer_crudeprev','arthritis_crudeprev',
    'highchol_crudeprev'
]

# Predictor columns: 25 behavioral, preventive, social determinant features
predictor_cols = [ 
    # Preventive care
    'access2_crudeprev','checkup_crudeprev','cholscreen_crudeprev',
    'colon_screen_crudeprev','dental_crudeprev','mammouse_crudeprev',

    # Behavioral risks
    'binge_crudeprev','csmoking_crudeprev','lpa_crudeprev','sleep_crudeprev',

    # Mental & physical health
    'depression_crudeprev','mhlth_crudeprev','phlth_crudeprev',
    'isolation_crudeprev','emotionspt_crudeprev',

    # Social determinants
    'foodinsecu_crudeprev','foodstamp_crudeprev','housinsecu_crudeprev',
    'shututility_crudeprev','lacktrpt_crudeprev',

    # Disability indicators
    'disability_crudeprev','selfcare_crudeprev','indeplive_crudeprev',

    # Demographics
    'totalpopulation','totalpop18plus'
]

print("Number of target columns:", len(target_cols))
print("Number of predictor columns:", len(predictor_cols))



# Output: Group predictors by domain


groups = {
    "Preventive Care Predictors": [
        'access2_crudeprev','checkup_crudeprev','cholscreen_crudeprev',
        'colon_screen_crudeprev','dental_crudeprev','mammouse_crudeprev'
    ],
    "Behavioral Risk Predictors": [
        'binge_crudeprev','csmoking_crudeprev','lpa_crudeprev','sleep_crudeprev'
    ],
    "Mental & Physical Health Predictors": [
        'depression_crudeprev','mhlth_crudeprev','phlth_crudeprev',
        'isolation_crudeprev','emotionspt_crudeprev'
    ],
    "Social Determinant Predictors": [
        'foodinsecu_crudeprev','foodstamp_crudeprev','housinsecu_crudeprev',
        'shututility_crudeprev','lacktrpt_crudeprev'
    ],
    "Disability & Functional Limitations": [
        'disability_crudeprev','selfcare_crudeprev','indeplive_crudeprev'
    ],
    "Demographic Predictors": [
        'totalpopulation','totalpop18plus'
    ]
}

print("\n=== Predictor Groups ===\n")
for group_name, columns in groups.items():
    print(group_name + " (" + str(len(columns)) + "):")
    for col in columns:
        print("  -", col)
    print()


In [ ]:
# STEP 4 — MEAN VALUE IMPUTATION

# Select only predictor + target columns for modeling
df_model = df_original[predictor_cols + target_cols].copy()

# Initialize mean-value imputer for numerical columns
imputer = SimpleImputer(strategy="mean")

# Apply imputation and rebuild DataFrame
df_model_imputed = pd.DataFrame(
    imputer.fit_transform(df_model),
    columns=df_model.columns
)

# output: total missing values after imputation
print("Total missing values after imputation:", df_model_imputed.isna().sum().sum())

# Show first 5 rows to verify the imputation worked correctly
df_model_imputed.head()


- This cell selects the 25 predictor columns and 10 target columns, then replaces missing values using mean imputation.
Since the PUBLIC dataset MAY contains some missing entries and formatting inconsistencies, imputation is required before scaling and model training.

In [ ]:
# STEP 5 — EXTRACT X AND y

# Extract predictor matrix (25 predictors)
X_base = df_model_imputed[predictor_cols]

# Extract target matrix (10 chronic diseases)
y = df_model_imputed[target_cols]

# Output
print("X_base shape (predictors):", X_base.shape)
print("y shape (targets):", y.shape)

# Display the first few rows of both
print("\nPreview of X_base (predictors):")
display(X_base.head())

print("\nPreview of y (targets):")
display(y.head())


This cell separates the imputed dataset into:

- X_base -> the 25 predictor variables

- y -> the 10 chronic disease target variables


These matrices will be used for all Phase 1 regression models and Phase 2 neural network modeling.

In [ ]:
# STEP 6 — STANDARDIZE PREDICTORS

# Initialize StandardScaler
scaler = StandardScaler()

# Fit on full predictor matrix and transform
X_scaled = scaler.fit_transform(X_base)

# Output
print("Shape of scaled predictors:", X_scaled.shape)
print("\nFirst 5 rows of scaled X (as array):")
print(X_scaled[:5])


- Many machine learning models perform better when predictors are normalized.
In this step, we apply StandardScaler, which transforms each predictor to:
The standardized value is computed as 

         z= (x-Mean)/ sigma

- This ensures all predictors are on a comparable scale before training Phase 1 models and the Phase 2 neural network.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Dictionary to store best params and scores
rf_best_params = {}
rf_best_scores = {}

# Parameter grid (safe & efficient)
param_grid = {
    "n_estimators": [150, 250, 350],
    "max_depth": [8, 12, 16],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2, 3],
    "max_features": ["sqrt", "log2"]
}

for disease in target_cols:
    print(f"\n$ Running GridSearchCV for {disease} ...")

    y_target = df_model_imputed[disease].values  # target column
    
    rf = RandomForestRegressor(random_state=42, n_jobs=-1)

    grid = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=3,                 # 3-fold CV
        scoring="r2",         # use R² for regression
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_scaled, y_target)

    # Save results
    rf_best_params[disease] = grid.best_params_
    rf_best_scores[disease] = grid.best_score_

    print(f"-> Best R²: {grid.best_score_:.4f}")
    print("-> Best Params:", grid.best_params_)

print("\n GRID SEARCH COMPLETE!")


In [ ]:
# ============================================================
# REFIT RANDOM FOREST MODELS USING GRIDSEARCH BEST PARAMS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

rf_final_models = {}          # store final fitted models
rf_final_importances = {}     # store importance vectors
rf_final_metrics = []         # store evaluation metrics

# Scale predictors once
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model_imputed[predictor_cols])

for disease in target_cols:

    print(f"Refitting tuned RF for: {disease}")

    # Target
    y = df_model_imputed[disease].values

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )

    # Load best params from GridSearchCV
    best_params = rf_best_params[disease]

    # Build tuned RF
    model = RandomForestRegressor(
        **best_params,
        random_state=42,
        n_jobs=-1
    )

    # Fit
    model.fit(X_train, y_train)

    # Store model + importances
    rf_final_models[disease] = model
    rf_final_importances[disease] = model.feature_importances_

    # Compute metrics
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    rf_final_metrics.append({
        "Disease": disease,
        "Train R2": r2_score(y_train, y_pred_train),
        "Test R2": r2_score(y_test, y_pred_test),
        "Train MAE": mean_absolute_error(y_train, y_pred_train),
        "Test MAE": mean_absolute_error(y_test, y_pred_test),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Best Params": best_params
    })

df_rf_tuned_results = pd.DataFrame(rf_final_metrics)
df_rf_tuned_results


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Copy tuned results (for safety)
df = df_rf_tuned_results.copy()

sns.set(style="whitegrid")

# ============================================================
# 1. TRAIN vs TEST R² — BAR CHART
# ============================================================

plt.figure(figsize=(14, 6))
df_r2 = df.melt(id_vars="Disease", value_vars=["Train R2", "Test R2"],
                var_name="Metric", value_name="R2 Score")

sns.barplot(data=df_r2, x="Disease", y="R2 Score", hue="Metric")
plt.xticks(rotation=45)
plt.title("Train vs Test R² (Tuned Random Forest)", fontsize=16)
plt.tight_layout()
plt.show()


# ============================================================
# 2. TRAIN vs TEST MAE — BAR CHART
# ============================================================

plt.figure(figsize=(14, 6))
df_mae = df.melt(id_vars="Disease", value_vars=["Train MAE", "Test MAE"],
                 var_name="Metric", value_name="MAE Score")

sns.barplot(data=df_mae, x="Disease", y="MAE Score", hue="Metric")
plt.xticks(rotation=45)
plt.title("Train vs Test MAE (Tuned Random Forest)", fontsize=16)
plt.tight_layout()
plt.show()


# ============================================================
# 3. TRAIN vs TEST RMSE — BAR CHART
# ============================================================

plt.figure(figsize=(14, 6))
df_rmse = df.melt(id_vars="Disease", value_vars=["Train RMSE", "Test RMSE"],
                  var_name="Metric", value_name="RMSE Score")

sns.barplot(data=df_rmse, x="Disease", y="RMSE Score", hue="Metric")
plt.xticks(rotation=45)
plt.title("Train vs Test RMSE (Tuned Random Forest)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# PRINT TOP 10 FEATURES FOR EACH TARGET (TUNED RF MODELS)
# ============================================================

print("\n===== TOP 10 FEATURES PER TARGET (TUNED RANDOM FOREST) =====\n")

for disease in target_cols:

    importance_vector = rf_final_importances[disease]   # feature importances for this disease

    # Pair predictors with importance scores
    pairs = list(zip(predictor_cols, importance_vector))

    # Sort descending
    pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)

    # Print output
    print(f"===== {disease.upper()} =====")
    for feature, score in pairs_sorted[:10]:
        print(f"{feature:30s} -> {score:.6f}")
    print()


In [ ]:
# ============================================================
# CLEAN CORRELATION PLOT FOR A SINGLE DISEASE (USING ONLY 25 PREDICTORS)
# ============================================================

import seaborn as sns
import matplotlib.pyplot as plt

# ---- choose your target disease here ----
target = "diabetes_crudeprev"        # change this to any disease

# restrict correlation to the 25 predictors + this target
cols_for_corr = predictor_cols + [target]

# compute correlation matrix
corr_matrix = df_model_imputed[cols_for_corr].corr()

# extract correlations with the target (excluding the target itself)
corr = corr_matrix[target].drop(target)

# sort values
corr_sorted = corr.sort_values(ascending=False)

# PLOT
plt.figure(figsize=(10, 14))
sns.barplot(
    x=corr_sorted.values,
    y=corr_sorted.index,
    hue=corr_sorted.values,
    palette="coolwarm",
    dodge=False,
    legend=False
)

plt.title(f"Correlation of Predictors with {target.upper()}", fontsize=16)
plt.xlabel("Correlation")
plt.ylabel("Predictor")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


### Validation of Random Forest Feature Importances Using Correlation Analysis (Diabetes Example)

To validate the Random Forest (RF) feature-importance results for diabetes_crudeprev, we compared RF rankings with Pearson correlations between diabetes prevalence and all behavioral, social, and preventive-care predictors. This joint analysis confirms whether RF aligns with known linear patterns and captures nonlinear or interaction-driven effects.

### 1. Strong Alignment Between Correlation and Random Forest Importance

The correlation analysis identifies several predictors with strong positive linear associations with diabetes, including:

- selfcare_crudeprev

- phlth_crudeprev (poor physical health)

- lpa_crudeprev (lack of physical activity)

- disability_crudeprev

- indeplive_crudeprev

- csmoking_crudeprev

These same predictors rank highly in the RF model, showing that RF correctly captures the strongest linear drivers of diabetes.

Most importantly:

- selfcare_crudeprev is both:

- the strongest Pearson correlate, and

- the top RF predictor.

This one-to-one alignment strongly validates the accuracy and reliability of Random Forest for identifying important features.

### 2.Why Some RF-Important Features Show Only Moderate or Weak Correlation

Some predictors have weaker linear correlation but receive higher RF importance, such as:

- binge_crudeprev

- checkup_crudeprev

- cholscreen_crudeprev

- mhlth_crudeprev

This is expected because:

- Correlation only measures linear relationships.

- RF captures nonlinear effects, thresholds, and predictor–predictor interactions.

Example:
- binge_crudeprev alone may not correlate strongly with diabetes, but diabetes risk increases sharply when binge drinking appears together with poor self-care, disability, or inactivity.
These combined effects are invisible in correlation but easily learned by Random Forest, reinforcing RF’s superiority in feature discovery.

### Why Socioeconomic Predictors Show Higher Correlation but Lower RF Importance
Predictors like 
- food insecurity,
- housing instability,
- lack of transportation, and
- utility shutoff
often show meaningful linear correlations with diabetes.
However, RF assigns them lower importance because:

They influence diabetes indirectly,

Their effects are mediated through behavior and preventive-care factors (e.g., lower physical activity, poor diet, missed checkups).

Thus, RF correctly prioritizes proximal behavioral and clinical predictors over distant socioeconomic drivers. This matches established public-health evidence.


### Overall Interpretation: Correlation + RF Importance Tell a Coherent Story

Where Correlation and RF Agree

- Key behavioral and health-status factors (self-care difficulty, disability, physical inactivity, poor physical health).

Where Random Forest Adds Value

- Revealing nonlinear and interaction-based influences (binge drinking, checkup frequency, cholesterol screening).

Where Correlation Adds Value

- Highlighting indirect socioeconomic stressors that RF ranks lower because they act through more immediate behavioral pathways.

#### Final Verdict: RF Is Highly Successful for Feature Importance

- This analysis clearly shows that Random Forest is a powerful and trustworthy method for identifying the most meaningful predictors of diabetes.
- Its ability to capture both linear patterns and complex nonlinear interactions makes RF superior for feature-importance analysis in chronic-disease modeling.

# Why We Tuned Random Forest Separately for Each Disease
- (Instead of Using One Fixed Parameter Set for All 10 Targets)
  In the initial Phase 1 modeling, we used one common set of Random Forest parameters for all diseases:

*RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)*
This one-size-fits-all assumption implies that:

- all diseases have similar predictor relationships,

- all require the same tree depth,

- all respond similarly to feature subsampling,

- all have the same noise level,

- all exhibit similar nonlinear patterns.

#### However, this assumption is incorrect.
*Each condition has different behavioral, medical, and socioeconomic determinants.*
*Thus forcing a single model structure on all diseases leads to overfitting and inaccurate feature importance.*

### Problems With the Original (Untuned) RF Models
1. RMSE and MAE Show Evidence of Overfitting

The original models reported:

Train RMSE extremely low (e.g., 0.16–0.30)

Train MAE extremely low (e.g., 0.05–0.40)

But on the held-out test set:

Test RMSE was 3–5× higher

Test MAE nearly tripled for some diseases

Examples:

### Random Forest Regression Performance Summary

| Disease           | Train RMSE | Test RMSE | Gap   |
|-------------------|------------|-----------|--------|
| Diabetes          | ~0.28      | ~0.76     | Large  |
| Obesity           | ~0.85      | ~2.29     | Huge   |
| High Cholesterol  | ~0.57      | ~1.55     | Large  |

This train–test gap indicates the model memorized training patterns rather than learning generalizable relationships.

2. Train R² Was Near-Perfect, but Test R² Dropped Significantly

Train R² values were between:

- 0.98–0.996 (too high for real-world health data)

But test R² values dropped to:

- 0.88–0.96, depending on the disease

- This gap again reflects overfitting.

3. Feature Importance Was Unstable and Unrealistic (which we thought we got to identify stronger relationships but they are not true)

Untuned RF produced:

- extremely high dominance of one or two predictors (e.g., self-care = 0.72)

- inconsistent patterns across similar diseases

- weaker agreement with correlation structure

- This instability made interpretation unreliable.

# What Tuned Random Forest Does Differently

Using GridSearchCV, we found the best hyperparameters for each disease independently.
This means each disease receives an optimized RF model.

GridSearch selected different values of:

- max_depth

- min_samples_leaf

- min_samples_split

- max_features

- n_estimators

Examples of Differences

- Diabetes preferred deeper trees and more estimators.

- COPD required shallower trees and stronger regularization.

- High cholesterol performed best with max_features="log2".

- BPHIGH benefited from larger leaf sizes and more splits.

*This confirms that no single parameter configuration is universally optimal.*


# Why the Tuned RF Model (GridSearchCV Per Disease) Is better

Using GridSearchCV, each disease received its own best-performing hyperparameters, improving model behavior substantially.

Improvement in RMSE and MAE After Tuning

Across nearly all diseases:

- Train RMSE increased slightly (as expected)

        = less memorization
        = healthier model behavior

- Test RMSE decreased or remained stable

        = better generalization
        = more reliable predictions

### Diabetes — Before vs After Tuning

| Metric      | Before Tuning | After Tuning | Interpretation                    |
|-------------|----------------|---------------|-----------------------------------|
| Train RMSE  | 0.27           | 0.49          | Good — reduces overfitting        |
| Test RMSE   | 0.75           | 0.79          | Very similar — stable performance |

### Obesity — Before vs After Tuning

| Metric      | Before Tuning | After Tuning | Interpretation                    |
|-------------|----------------|---------------|-----------------------------------|
| Train RMSE  | 0.85           | 1.61          | Good — reduces overfitting        |
| Test RMSE   | 2.29           | 2.43          | Very similar — stable performance |

This shows:

- The model is no longer memorizing training data

- Generalization remains strong

- The train–test gap is more realistic and scientifically credible

Improvement in Train–Test MAE Gap

- Before tuning:

        Train MAE was extremely low (e.g., 0.05–0.40)

        Test MAE was 3× higher (e.g., 0.9–2.3)

- After tuning:

        Train MAE increased (less overfitting)

        Test MAE stayed stable or improved slightly

        Gap between train and test MAE became smaller

This confirms the tuned model is less biased, and more generalizable.

Because tuned RF models generalize better:

- feature importance becomes more stable,

- rankings become more medically interpretable,

- nonlinear effects (e.g., interactions) are captured more reliably.

This is why, after tuning:

- poor physical health -> CHD, stroke, COPD

- routine checkups -> hypertension, arthritis

- cholesterol screening -> high cholesterol

- self-care disability -> diabetes

we train a separate Random Forest Regressor for each of the 10 chronic disease outcomes.
Random Forests naturally provide feature importance scores, which allow us to identify the strongest predictors driving each disease.

#### This step answers RQ2: Which predictors most influence each chronic condition?


## Medical Interpretation of Top Predictors Identified by the Random Forest Model
---

### Medical Validation of Random Forest Feature Importance

The high-ranking predictors selected by the Tuned Random Forest model show strong agreement with established epidemiological and clinical research, confirming that the model is capturing medically meaningful relationships rather than statistical noise.

## 1. Diabetes (DIABETES_CRUDEPREV) and Self-Care Disability (selfcare_crudeprev)

The model identifies self-care disability as the strongest predictor of diabetes prevalence (importance ≈ 0.196).

Clinical and Epidemiological Evidence
#### a. Self-Care & Glycemic Control

Diabetic neuropathy reduces the ability to perform daily tasks, causing a cycle where poor self-care worsens diabetes.

Impact of neuropathy on adherence to diabetes-related self-care activities
https://pmc.ncbi.nlm.nih.gov/articles/PMC4936822/

#### b. Diabetes as a Major Cause of Disability

Diabetes complications (vision loss, neuropathy, amputation) significantly increase disability risk.

Disability and Diabetes in Adults – Diabetes in America
https://www.ncbi.nlm.nih.gov/books/NBK609951/

#### c. Bidirectional Relationship

Self-care disability both predicts diabetes and results from diabetes progression.

Prevalence of Diagnosed Diabetes Among U.S. Adults With Disabilities (CDC)
https://diabetesjournals.org/care/article/48/3/324/157646/

Interpretation

High prevalence of self-care difficulty signals a population with severe, long-standing diabetes, making this a medically logical top predictor.

## 2. Obesity (OBESITY_CRUDEPREV) and Current Smoking (csmoking_crudeprev)

Current smoking is the strongest predictor of obesity prevalence (importance ≈ 0.230).

Clinical Evidence
#### a. Clustering of Risk Behaviors

Smoking and obesity cluster in socioeconomically disadvantaged groups.

Socioeconomic Status, Smoking, Alcohol Use, Physical Activity, and Diet as Determinants of Obesity
https://pmc.ncbi.nlm.nih.gov/articles/PMC4948154/

#### b. Shared Environmental & Social Stressors

Smokers in low-SES neighborhoods show higher BMI.

Mechanisms Linking Socioeconomic Disadvantage and BMI in Smokers
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3903428/

#### c. Post-Cessation Weight Gain

Smoking cessation causes 4–5 kg weight gain on average.

Systematic review & meta-analysis (63,403 quitters)
https://pubmed.ncbi.nlm.nih.gov/26114839/

Meta-analysis of smoking cessation and weight gain
https://pubmed.ncbi.nlm.nih.gov/22782848/

#### Interpretation

RF correctly detects that smoking is deeply intertwined with obesity due to behavioral clustering and weight gain after cessation.

## 3. High Blood Pressure (BPHIGH_CRUDEPREV) and Routine Checkups (checkup_crudeprev)

Routine checkups are the strongest predictor of diagnosed hypertension prevalence (importance ≈ 0.155).

Medical Evidence
#### a. Hypertension Requires Screening to Diagnose

Hypertension is asymptomatic; routine BP checks are needed.

2017 ACC/AHA Hypertension Clinical Guideline
https://www.ahajournals.org/doi/10.1161/HYP.0000000000000065

Interpretation

More checkups = more opportunities for hypertension detection.
RF correctly identifies healthcare access as the main determinant of diagnosed hypertension.

## 4. High Cholesterol (HIGHCHOL_CRUDEPREV) and Cholesterol Screening (cholscreen_crudeprev)

Cholesterol screening is the strongest predictor of diagnosed high cholesterol (importance ≈ 0.145).

Clinical Evidence

USPSTF Recommendation on Lipid Screening
https://www.uspreventiveservicestaskforce.org/uspstf/recommendation/statin-use-in-adults-preventive-medication

Interpretation

High cholesterol is asymptomatic; RF correctly learns that diagnosis is driven entirely by screening rates.

## 5. COPD (COPD_CRUDEPREV) and Poor Physical Health (phlth_crudeprev)

Poor physical health days are the top predictor of COPD prevalence (importance ≈ 0.211).

Clinical Evidence

GOLD 2023 Global COPD Report
https://goldcopd.org/2023-gold-report-2/

Interpretation

COPD causes severe fatigue, dyspnea, and exercise limitation—exactly what “poor physical health days” measure.
RF correctly links functional impairment to COPD prevalence.

## 6. Cancer (CANCER_CRUDEPREV) and Colorectal Screening (colon_screen_crudeprev)

Colorectal screening is the strongest predictor (importance ≈ 0.153).

Clinical Evidence

USPSTF Colorectal Cancer Screening Guidelines
https://www.uspreventiveservicestaskforce.org/uspstf/recommendation/colorectal-cancer-screening

Interpretation

Screening increases diagnosis of cancers and pre-cancers.
RF logically identifies screening rates as the strongest driver of cancer prevalence.

## 7. CHD (CHD_CRUDEPREV) and Poor Physical Health (phlth_crudeprev)

Poor physical health days are the strongest predictor of coronary heart disease (importance ≈ 0.187).

Evidence

Self-Rated Health and Cardiovascular Disease (HRS Study)
Poor physical health is both a cause and consequence of CHD.

Interpretation

CHD severely impacts physical function, so RF is correctly picking up the symptomatic burden of the disease.

## 8. Stroke (STROKE_CRUDEPREV) and Poor Physical Health (phlth_crudeprev)

Poor physical health days are the strongest predictor of stroke prevalence (importance ≈ 0.192).

Evidence

Long-Term Disability After Stroke (AVAIL Registry)
Stroke survivors exhibit major functional decline and disability.

Interpretation

RF captures the lasting physical sequelae (“poor physical health days”) of stroke survivors.

## 9. Arthritis (ARTHRITIS_CRUDEPREV) and Routine Checkups (checkup_crudeprev)

Routine checkups are the strongest predictor (importance ≈ 0.190).

Clinical Evidence

Role of Primary Care in Management of Rheumatoid Arthritis
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6449174/

Interpretation

Arthritis requires lifelong medical management, so routine checkups are a strong proxy for disease prevalence.

## Summary: Model Validation
 Clinically Valid

Random Forest feature rankings align closely with known risk factors, symptoms, and detection pathways.

 Captures Real Mechanisms

Direct risk factors (e.g., smoking -> COPD/obesity)

Disease consequences (e.g., stroke -> poor physical health)

Screening/detection effects (e.g., checkups -> hypertension)

 Medically Interpretable

Each top feature is backed by peer-reviewed evidence and established epidemiology.

In [ ]:
rf_best_params

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

gb_best_params = {}
gb_best_scores = {}

gb_param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
}

print(">> Starting GridSearchCV for Gradient Boosting...\n")

for disease in target_cols:
    print(f"Running GridSearch for {disease} ...")
    
    y = df_model_imputed[disease].values
    X = X_scaled

    gb = GradientBoostingRegressor(random_state=42)

    grid = GridSearchCV(
        gb,
        gb_param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X, y)

    gb_best_params[disease] = grid.best_params_
    gb_best_scores[disease] = grid.best_score_

    print(f"$ Best R²: {grid.best_score_:.4f}")
    print(f"$ Best Params: {grid.best_params_}\n")

print("* Gradient Boosting tuning complete!")


In [ ]:
# ============================================================
# REFIT GRADIENT BOOSTING MODELS USING GRIDSEARCH BEST PARAMS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import pandas as pd

gb_final_models = {}
gb_final_metrics = []

print("-> Re-fitting all Gradient Boosting models with tuned parameters...\n")

for disease in target_cols:

    print(f"->Re-fitting GB for: {disease}")

    # Target values
    y = df_model_imputed[disease].values
    X = X_scaled  # already scaled in RF step

    # Train–Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # Load best parameters found by GridSearch
    best_params = gb_best_params[disease]

    # Create tuned Gradient Boosting model
    gb = GradientBoostingRegressor(
        **best_params,
        random_state=42
    )

    # Fit model
    gb.fit(X_train, y_train)

    # Store model
    gb_final_models[disease] = gb

    # Predictions
    y_pred_train = gb.predict(X_train)
    y_pred_test  = gb.predict(X_test)

    # Metrics
    gb_final_metrics.append({
        "Disease": disease,
        "GB Train R2": r2_score(y_train, y_pred_train),
        "GB Test R2": r2_score(y_test, y_pred_test),
        "GB Train MAE": mean_absolute_error(y_train, y_pred_train),
        "GB Test MAE": mean_absolute_error(y_test, y_pred_test),
        "GB Train RMSE": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "GB Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "Best Params": best_params
    })

print("\n$ All Gradient Boosting models evaluated.")

df_gb_results = pd.DataFrame(gb_final_metrics)
df_gb_results


In [ ]:
plt.figure(figsize=(12, 6))
df_gb_results.set_index("Disease")[["GB Train R2", "GB Test R2"]].plot(kind="bar", figsize=(12,6))
plt.title("Gradient Boosting: Train vs Test R²")
plt.ylabel("R² Score")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
df_gb_results.set_index("Disease")[["GB Train MAE", "GB Test MAE"]].plot(kind="bar", figsize=(12,6))
plt.title("Gradient Boosting: Train vs Test MAE")
plt.ylabel("MAE")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
df_gb_results.set_index("Disease")[["GB Train RMSE", "GB Test RMSE"]].plot(kind="bar", figsize=(12,6))
plt.title("Gradient Boosting: Train vs Test RMSE")
plt.ylabel("RMSE")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# FINAL ANALYSIS FOR PHASE 1

Random Forest is the superior model for:

- Interpreting feature importance

- Identifying key drivers of chronic diseases

- Stability and reliability across all 10 diseases

- Alignment with real-world epidemiology

Gradient Boosting:

- Performs similarly in prediction

- Offers no major improvement over RF

- But is less interpretable and more sensitive to noise

Therefore, RF remains the primary model for this project.

### Model Comparison Summary: Why Random Forest Is the Preferred Model

Although both the Tuned Random Forest (RF) and Tuned Gradient Boosting (GB) models achieved strong predictive performance across all ten chronic diseases, a deeper analysis reveals that Random Forest is the more appropriate primary model for this project, especially for feature importance interpretation and public-health decision-making.

1. Predictive Performance: RF and GB Perform Similarly

Across all diseases:

- Test R² values for RF and GB differ only by 0.01–0.02 in most cases

- MAE and RMSE values remain close, with no consistent or meaningful advantage for Gradient Boosting

- Both models exhibit high generalization ability, with stable train–test performance gaps

### Conclusion:

RF and GB are equally strong in prediction quality.
Therefore, predictive accuracy alone does not justify selecting GB over RF.

2. Random Forest Provides Superior Feature Importance Stability

A key goal of this project is not only to predict chronic disease prevalence, but also to identify the leading behavioral, social, and preventive-care factors that drive each disease.

Random Forest offers clear advantages here:

- RF feature importance is stable across runs

Gradient Boosting importances are more volatile because boosting magnifies small residual patterns.

- RF importance aligns more strongly with:

        Pearson correlation patterns

        Epidemiological literature

        Clinical risk pathways
Examples:

- Self-care disability -> diabetes

- Smoking -> COPD/obesity clusters

- Poor physical health -> CHD & stroke

- Screening/checkups -> diagnosed conditions

These pathways were captured more cleanly and consistently in RF.

Conclusion:

For interpretability, stability, and medical alignment, Random Forest is clearly superior.

4. Practical Reason: Interpretability > Marginal Gain

In public health applications, the goal is not only “high prediction,” but understanding what drives disease risk so interventions can be targeted.

Gradient Boosting is a powerful predictive model,
but its feature contributions are less intuitive to interpret and harder to justify in a scientific or policy context.

- Random Forest strikes the perfect balance:

- High predictive accuracy

- Strong interpretability

- Clinically coherent feature importance

Conclusion:

Even if GB achieved slightly higher R² in a few diseases, RF remains the more scientifically valuable model because it explains chronic disease risk more reliably and clearly.

### Final Decision for This Project
- Use Random Forest as the primary model

- for identifying key chronic disease predictors and producing interpretable feature-importance results.

- Gradient Boosting can be reported as a secondary validation model

- to show that “another advanced method confirms similar predictive power,”
strengthening confidence in your RF-based findings.
-----

# PHASE 2
## **Introduction to the Chronic Health Risk Index (CHRI)**

The CDC PLACES dataset provides prevalence estimates for ten major chronic 
conditions at the census-tract level (such as diabetes, COPD, hypertension, and 
stroke). Although each condition is important on its own, communities do not 
experience these diseases independently. Chronic conditions frequently **co-occur**, 
share risk factors, and reinforce one another.

To capture this multidimensional disease burden, we construct a **Chronic Health 
Risk Index (CHRI)** — a single composite score summarizing overall chronic disease 
load for each census tract.


---

## **How We Learn CHRI from the Data**

Since CHRI is not provided in the dataset, we train a neural network model to 
*learn* a representation of disease burden.

- **Input:** the ten chronic disease prevalence values  
- **Training target:** the mean of the ten diseases  
- **Output:** a single CHRI score  

Using the mean disease burden as a target provides a grounded, interpretable 
signal representing *overall chronic disease load*, while allowing the neural 
network to learn deeper structure within the data.

This approach transforms the model into a **representation-learning mechanism** 
that captures how diseases interact and cluster across populations.

---

## **Why a Neural Network?**

The neural network is not used for classification here — instead, it is used as 
a **nonlinear feature extractor** to learn a continuous “health risk score.”

A simple arithmetic mean cannot capture:

- nonlinear interactions between diseases,  
- compounding effects (e.g., diabetes + hypertension),  
- threshold or saturation behaviors,  
- disease clustering patterns observed in epidemiology.

A neural network offers:

- the ability to learn **nonlinear patterns** among diseases,  
- the ability to represent **shared comorbidity structure**,  
- the flexibility to learn a **smooth continuous index** that generalizes across tracts.

This produces a CHRI metric that is more realistic and behaviorally aligned with 
how chronic illness accumulates at the community level.

---

## **How CHRI Is Used in Phase 2**

After the neural network learns CHRI values:

1. CHRI is **scaled from 0 to 100** for interpretability.  
2. Three CHRI tiers (Low / Medium / High) are derived using quantiles.  
3. These tiers are used in downstream analysis and classification.

CHRI therefore becomes the foundation for understanding and predicting chronic 
health risk at the census-tract level.

---

## **Summary**

CHRI provides:

- A **data-driven**, interpretable representation of chronic disease burden  
- A way to combine ten disease metrics into one meaningful score  
- A foundation for classification models that predict community-level health risk  
- A method aligned with epidemiological reality, where chronic illnesses co-occur 
  and accumulate in predictable patterns  

This approach ensures that the CHRI score is grounded, interpretable, and 
statistically informed, while also capturing nonlinear relationships present in 
real-world population health data.


## **Why Do We Need a Chronic Health Risk Index (CHRI)?**

The CDC PLACES dataset contains ten separate chronic disease prevalence measures 
(diabetes, hypertension, obesity, COPD, cancer, etc.).  
Analyzing these diseases **one-by-one** provides useful insight, but it does not capture 
a fundamental public-health reality:

### **Chronic diseases do not occur in isolation.**
Communities that experience high diabetes prevalence often also show elevated:
- hypertension,
- obesity,
- stroke risk,
- coronary heart disease,
- COPD.

This phenomenon — known as **multimorbidity clustering** — is widely documented in 
population health research. It reflects shared behavioral, environmental, and 
socioeconomic risk factors.

Because of this, public health agencies often rely on **composite health indices** 
(e.g., Social Vulnerability Index, Community Health Scores, Deprivation Index) rather 
than individual disease metrics alone.

CHRI follows the same principle:  
it provides **one unified score** that summarizes the overall chronic disease burden of a census tract.


## ***(Initial Attempt): Linear CHRI Neural Network***

Our first implementation of the CHRI neural network used a *single* linear layer:

\[
\text{CHRI} = w_1 x_1 + \dots + w_{10} x_{10} + b
\]

This architecture is mathematically equivalent to a **linear regression** model.  
The motivation was to start with the simplest possible composite index that respects interpretability:

- Each chronic disease receives a direct weight.
- CHRI is a weighted sum of disease prevalence.
- The model is transparent and easy to explain.

### Limitations of the Linear Model

However, after implementing this version and discussing with the instructor, several limitations became clear:

1. **Linearity assumption**
   - The model assumes that all chronic diseases combine in a purely *additive* linear fashion.
   - This ignores known non-linear relationships between conditions (e.g., diabetes × obesity interactions).

2. **No capacity for interaction effects**
   - A single linear layer cannot capture how diseases may jointly amplify risk.
   - Multimorbidity research shows strong compounding effects that require nonlinear modeling.

3. **Over-simplification of multimorbidity**
   - Chronic conditions share complex biological and behavioral pathways.
   - A linear-only representation is too restrictive to represent real-world chronic disease burden.

4. **Poor representation learning**
   - The linear model is not capable of learning a meaningful *latent* representation.
   - CHRI becomes nothing more than a weighted average.

## Why the Linear CHRI Model Was Not Sufficient

Although the linear CHRI_NN successfully trained, it became clear that we could not proceed with this approach for three major reasons:

### 1. Linear outputs = no added value beyond the baseline mean
The model could only learn a weighted sum of its inputs.  
Because our target was the **mean** of the 10 crude prevalences, the linear NN can only learn a transformation almost identical to that mean.

As a result:

- The linear model does not extract new structure.
- It cannot represent latent chronic health burden.
- It cannot capture any meaningful nonlinear interactions.

### 2. Data exploration revealed complex relationships
Our correlation heatmap showed:

- Strong correlations between some diseases (e.g., diabetes–stroke),
- Weak or negative correlations for others (e.g., obesity–cancer),
- Heterogeneous dependency patterns.

This suggested that real multimorbidity patterns are **not purely linear**, making a single-layer model inappropriate.




In [ ]:
# ============================================================
# CORRELATION HEATMAP FOR ALL 10 CHRONIC DISEASE VARIABLES
# ============================================================

import seaborn as sns
import matplotlib.pyplot as plt

# Select only the 10 chronic disease columns
disease_df = df_model_imputed[target_cols]

# Compute correlation matrix
corr_matrix = disease_df.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8},
    fmt=".2f"
)

plt.title("Correlation Heatmap of Chronic Disease Prevalence Variables", fontsize=16, pad=15)
plt.tight_layout()
plt.show()


### 3. Conceptual mismatch for a composite index
A composite health-risk index must be able to:

- capture nonlinear effects,  
- represent joint risk amplification,  
- compress multidimensional health patterns into a meaningful latent score.

The linear model cannot do this.

Therefore, we concluded that a **nonlinear neural network** was required to construct a valid, data-driven CHRI score.


## Redesigned Approach: Nonlinear CHRI Neural Network

To address the limitations of the linear model, we re-engineered CHRI_NN into a **multi-layer feedforward neural network** with hidden layers and ReLU activations:

- **Input layer:** 10 chronic disease prevalences  
- **Hidden Layer 1:** 32 neurons + ReLU  
- **Hidden Layer 2:** 16 neurons + ReLU  
- **Output Layer:** 1 CHRI score  

### Why this model works:

1. **Captures nonlinear interactions**  
   Conditions like diabetes and obesity jointly increase risk in a way that cannot be modeled linearly.

2. **Learns a latent health-risk representation**  
   Instead of a weighted sum, the NN extracts a richer underlying risk factor.

3. **Flexibility and scalability**  
   The model can be extended with more features or deeper layers in future work.

4. **Better aligned with public-health modeling**  
   Modern multimorbidity research uses nonlinear models, autoencoders, and deep embeddings—our redesigned CHRI follows this paradigm.

5. **Responds directly to instructor feedback**  
   The redesign ensures that the CHRI is not just a linear echo of the target mean, but a learned nonlinear health-index.

This improved NN architecture allowed us to continue developing CHRI in a scientifically valid and methodologically defensible way.


In [ ]:
# ---------------------------------------------------
# Rebuild y before running NN
# ---------------------------------------------------
y = df_model_imputed[target_cols].copy().values   # (n,10)


# ---------------------------------------------------
# Convert disease matrix into tensors
# ---------------------------------------------------
X_tensor = torch.tensor(y.astype("float32"))              # (n,10)
y_tensor = torch.tensor(y.mean(axis=1).astype("float32")) # (n,)
y_tensor = y_tensor.view(-1, 1)                            # (n,1)


# ---------------------------------------------------
# Define NONLINEAR Neural Network for CHRI
# ---------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim

class CHRIModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

model = CHRIModel()
print(model)   #  MODEL SUMMARY PRINTS HERE


# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal Trainable Parameters: {total_params:,}")


# ---------------------------------------------------
# Training setup
# ---------------------------------------------------
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

loss_history = []

# ---------------------------------------------------
# Training loop
# ---------------------------------------------------
for epoch in range(300):
    optimizer.zero_grad()
    pred = model(X_tensor)
    loss = criterion(pred, y_tensor)
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

print("\nCHRI_NN (nonlinear) training completed.")
print(f"Final Training Loss: {loss_history[-1]:.6f}")


In [ ]:
# Layer 1 weights & bias
w1 = model.fc1.weight.detach().numpy()
b1 = model.fc1.bias.detach().numpy()

# Layer 2 weights & bias
w2 = model.fc2.weight.detach().numpy()
b2 = model.fc2.bias.detach().numpy()

# Layer 3 weights & bias (final CHRI weights)
w3 = model.fc3.weight.detach().numpy()
b3 = model.fc3.bias.detach().numpy()

print("Layer 1 Weights:", w1)
print("Layer 1 Bias:", b1)

print("\nLayer 2 Weights:", w2)
print("Layer 2 Bias:", b2)

print("\nLayer 3 Weights (Final CHRI Weights):", w3)
print("Layer 3 Bias:", b3)


In [ ]:
# ============================================================
# PLOTTING TRAINING LOSS CURVE FOR NONLINEAR CHRI_NN
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

# Plot loss
plt.plot(loss_history, linewidth=2)

# Title & labels
plt.title("CHRI_NN Training Loss Curve (Nonlinear Model)", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)

# Nice grid
plt.grid(True, linestyle="--", alpha=0.5)

# Tight layout
plt.tight_layout()
plt.show()


*Interpretation of the Loss Curve*

- The loss decreased smoothly and consistently, indicating that
the network learned a stable mapping from disease prevalence → CHRI.

- No oscillation or upward drift is seen ->
optimizer is well-behaved and learning rate is appropriate.

*The final loss (~0.065) is very small relative to the scale of the inputs.*

In [ ]:
# FINAL CHRI CALCULATION

# PURPOSE:
# --------
# Convert the neural network output into a usable CHRI score (0–100),
# attach it to the cleaned dataset, and categorize each census tract
# into low / medium / high chronic health risk tiers.

# 1. Get CHRI values predicted by the NN
chri_nn = model(X_tensor).detach().numpy().flatten()

# 2. Scale CHRI to range 0–100
chri_scaled = (chri_nn - chri_nn.min()) / (chri_nn.max() - chri_nn.min())
chri_scaled *= 100  # now range is 0 to 100


# 3. Attach CHRI values to the dataset
df_clean = df_original.copy()
df_clean['CHRI'] = chri_scaled


# 4. Create risk tiers using quantiles
df_clean['CHRI_Tier'] = pd.qcut(
    df_clean['CHRI'],
    q=3,                     # 3 quantile groups
    labels=["low", "medium", "high"]
)

print("CHRI successfully added with tiers.")
df_clean[['CHRI', 'CHRI_Tier']].head()


### 1. Model Architecture Overview

The nonlinear CHRI model is a small feed-forward neural network designed to learn a single unified Chronic Health Risk Index (CHRI) from the 10 chronic disease prevalence rates.

Input (10 diseases)
   |
Fully Connected Layer (10 -> 32) + ReLU
   |
Fully Connected Layer (32 -> 16) + ReLU
   |
Output Layer (16 -> 1)

This architecture allows the model to learn:

- Nonlinear relationships between diseases

- Interaction patterns (e.g., diabetes x obesity effects)

- Shared latent disease burden factors

#### This is more expressive than the earlier linear CHRI model.

The model has:

- Total Trainable Parameters: 897
- The network learns intermediate representations, not just a direct weighting of the inputs

#### This is the mechanism by which the model captures nonlinear and interaction effects

In [ ]:
# CORRELATION OF CHRI WITH EACH DISEASE

chri_corr = {}

for disease in target_cols:
    corr_value = df_clean['CHRI'].corr(df_clean[disease])
    chri_corr[disease] = corr_value

# Convert to DataFrame for nicer display
chri_corr_df = pd.DataFrame.from_dict(chri_corr, orient='index', columns=['Correlation_with_CHRI'])
chri_corr_df.sort_values(by='Correlation_with_CHRI', ascending=False)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))

sns.barplot(
    x=chri_corr_df['Correlation_with_CHRI'],
    y=chri_corr_df.index,
    hue=chri_corr_df['Correlation_with_CHRI'],  # required for seaborn ≥ 0.14
    palette="coolwarm",
    dodge=False,
    legend=False
)

plt.title("Correlation of CHRI with Individual Chronic Diseases", fontsize=14)
plt.xlabel("Correlation")
plt.ylabel("Disease")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
df_clean[[*target_cols, "CHRI"]].corr()["CHRI"].sort_values(ascending=False)


In [ ]:
df_clean["CHRI"].hist(bins=30, figsize=(6,4))


# Interpreting CHRI Using Correlation Strength and Example Census Tracts

The CHRI neural-network model produces a single risk score summarizing the combined burden of 10 chronic diseases for each census tract. To understand how CHRI behaves, we interpret it in two complementary ways:

1. How Correlation Strength Influences CHRI

Each disease contributes to CHRI according to how consistently it moves with overall chronic burden:

- High-correlation diseases (strong influence on CHRI)

- Examples: Stroke, CHD, COPD, BP High, BP Med

These conditions often cluster together within communities due to shared risk factors (aging population, poor access to care, socioeconomic stress, etc.).
Their high positive correlation with CHRI means:

- When hypertension or stroke prevalence is high, CHRI almost always becomes high.

- These diseases act as anchor variables, exerting strong upward pressure on the CHRI score.

- Neural network weights learn these relationships automatically — even nonlinear effects.

Moderate-correlation diseases

- Examples: Diabetes, Obesity, Arthritis

These contribute meaningfully but with more variability.
- For example:

- Obesity may be high in a younger population with low CHD or stroke.

- Arthritis may be high without showing extreme cardiometabolic burden.

Thus, these variables push CHRI upward, but not as predictably.

Lower-correlation diseases

- Example: Cancer prevalence

- Cancer prevalence varies for reasons not always tied to chronic burden (screening access, demographic structure, historical exposures).
Therefore:

- Cancer influences CHRI, but less strongly, because its variance is not tightly coupled with other diseases.

In [ ]:
# ============================================================
# PICK EXAMPLE TRACTS: HIGH CHRI vs LOW CHRI
# ============================================================

# Select one example tract from each tier
high_example = df_clean[df_clean["CHRI_Tier"] == "high"].head(1)
low_example  = df_clean[df_clean["CHRI_Tier"] == "low"].head(1)

print("^ HIGH-RISK TRACT EXAMPLE")
display(high_example[target_cols + ["CHRI", "CHRI_Tier"]])

print("~ LOW-RISK TRACT EXAMPLE")
display(low_example[target_cols + ["CHRI", "CHRI_Tier"]])


### Interpretation of Example Census Tract (Low CHRI Tier)
Why This Tract Falls into the Low CHRI Tier

A CHRI score of 43.95 (0–100 scale) indicates relatively low combined chronic-disease burden. Several disease values support this classification:

1. Low Cardiometabolic Burden

- Diabetes (9.5%) is below the national average in many urban areas.

- Obesity (35.8%), while moderate, is still lower than high-risk regions which often exceed 40–45%.

- High cholesterol (30.1%) is also moderate.

2. Low Serious Disease Prevalence

- Stroke (2.4%) and COPD (5.1%) are both low—these are typically high in medically vulnerable tracts.

- CHD (4.8%) is also notably low.

These diseases are heavily weighted by the neural network because they represent severe clinical burden. Low values therefore substantially reduce the CHRI score.

3. Moderate but Not Alarming Arthritis (25.2%)

- Arthritis is fairly common nationally; 25% is not unusually high and does not strongly elevate overall risk.

4. High BPMED (75.5%) Indicates Good Access to Care

- A strikingly high value for BPMED (blood pressure medication adherence) suggests:

- Good healthcare access

- Strong chronic-disease management

- Higher likelihood of early detection and treatment

*This tends to reduce uncompensated disease risk, further lowering CHRI.*

In [ ]:
# Define bins and labels
bins = list(range(0, 110, 10))  # [0,10,20,30,...,100]
labels = [f"{i}-{i+10}" for i in range(0, 100, 10)]

# Create bucket column
df_clean["CHRI_Range"] = pd.cut(df_clean["CHRI"], bins=bins, labels=labels, right=False)

# Count values in each range
chri_range_counts = df_clean["CHRI_Range"].value_counts().sort_index()

chri_range_counts


In [ ]:
# EXTRACT LATITUDE & LONGITUDE (FIXED)


df_map = df_clean.copy()

# Extract longitude first value inside POINT(...)
df_map['lon'] = df_map['geolocation'].str.extract(r'POINT \(([-\d\.]+) ')[0].astype(float)

# Extract latitude second value inside POINT(...)
df_map['lat'] = df_map['geolocation'].str.extract(r'POINT \([-\d\.]+ ([\d\.]+)\)')[0].astype(float)

print("Latitude/Longitude extraction completed.")
df_map[['lon','lat']].head()



To create tract-level geospatial visualizations, we extract:

- **Longitude** -> the first number inside `POINT(...)`
- **Latitude** -> the second number inside `POINT(...)`

We use regular expressions (`str.extract`) to pull out both values and convert them to floating-point numbers.

This step prepares the dataset for mapping in later cells (e.g., Folium plots).


In [ ]:
# FINAL TRACT-LEVEL CHRI MAP


def tier_color(t):
    t = str(t).lower()
    return "green" if t=="low" else "yellow" if t=="medium" else "red"

m = folium.Map(location=[39.5, -98.35], zoom_start=4)
cluster = MarkerCluster().add_to(m)

for _, row in df_map.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color=tier_color(row['CHRI_Tier']),
        fill=True,
        fill_color=tier_color(row['CHRI_Tier']),
        fill_opacity=0.8,
        popup=f"CHRI: {row['CHRI']:.2f}"
    ).add_to(cluster)

m


## Final Tract-Level CHRI Map (Folium)

This cell visualizes the **tract-level CHRI values** across the United States using an
interactive Folium map.

###  This map shows:
- **Each census tract** is plotted as a small circular marker.
- Color indicates the risk tier:
  - GREEN **Low** CHRI  
  - ORANGE/YELLOW **Medium** CHRI  
  - RED **High** CHRI


###  Importance:
This visualization helps identify:
- Geographic clusters of high-risk areas  
- States or counties with widespread chronic-health concerns  
- Potential disparities between regions  
- Pockets of low-risk communities  

The map provides public-health teams and policymakers a spatial understanding of
chronic-health risk distribution.

The data used here comes directly from the **neural-network-computed CHRI scores**
and the **latitude/longitude extracted from the dataset**.


# RQ1 : Predictive Strength
How strongly do behavioral, preventive-care, and social-determinant predictors explain variation in chronic disease prevalence across U.S. census tracts?

To evaluate predictive strength, we trained two families of models across all ten chronic conditions:

Tuned Random Forest (RF) models

Tuned Gradient Boosting (GB) models

Both algorithms independently confirmed that behavioral, preventive-care, and social-determinant variables are highly predictive of chronic disease burden.

1. Strong Predictive Power Across Both RF and GB Models
Random Forest (RF) performance

Across all diseases:

- Test $R^2$ ranged from ~0.86 to ~0.98

- Train–test gaps were minimal

- Test MAE and RMSE were stable and low

Examples:

- Stroke -> $R^2 \approx 0.97$

- COPD -> $R^2 \approx 0.98$

- CHD -> $R^2 \approx 0.93$–0.95

- Obesity -> $R^2 \approx 0.87$–0.89

Gradient Boosting (GB) performance

The tuned GB models performed similarly or slightly better in some cases:

- Diabetes -> Test $R^2 = 0.95$

- CHD -> Test $R^2 = 0.96$

- Stroke -> Test $R^2 = 0.98$

- COPD -> Test $R^2 = 0.98$

The improvement of GB over RF was generally small — often only 0.01–0.03 increase in $R^2$.



2. Why Both Models Achieve High Predictive Power
- (a) Predictors represent core epidemiological pathways

        The variables include:

            Smoking rates

            Obesity/inactivity

            Disability and poor physical health

            Screening behaviors (checkup, cholesterol screening, colon cancer screening)

            Social determinants (access, socioeconomic strain)

All are directly tied to chronic disease prevalence in public health literature.

- (b) Nonlinear interactions are handled well

        Both RF and GB capture effects such as:

            Screening rates -> detection prevalence

            Disability -> worsening cardiometabolic outcomes

            Smoking -> multiple chronic diseases via common pathways

These relationships are highly nonlinear, and tree-based models excel at modeling them.

3. Interpretation of Predictive Strength

- Across the ten chronic diseases, both RF and GB demonstrate that the predictors explain the vast majority of variation.

- Conditions like stroke, COPD, CHD achieved extremely high predictability

- Even harder-to-model outcomes (e.g., obesity, high cholesterol) achieved strong $R^2 > 0.84$

This indicates:

- Chronic disease patterns across U.S. census tracts are deeply shaped by behavioral, preventive-care, and social-determinant factors.

The consistency across two different ML algorithms strengthens confidence in this conclusion.

4. Conclusion for RQ1

- Behavioral, preventive-care, and social determinants are explanatory of community-level chronic disease burden.
Across both RF and GB models:



# RQ2 : Feature Importance
### Which predictors most influence each chronic condition (e.g., diabetes, obesity, high blood pressure, coronary heart disease, stroke)?

To address this question, we used Tuned Random Forest (RF) models to extract feature importances for each of the ten chronic diseases. Unlike the earlier approach where the same hyperparameters were used for all targets, the tuned RF models learned target-specific optimal hyperparameters, providing more reliable and interpretable importance scores.

We validated the RF importance results using:

- Pearson correlation analysis

- Epidemiological and clinical literature



Together, these analyses provide a coherent and medically grounded explanation for the most influential predictors.

## 1. Importance of Using Tuned RF Models
Before tuning:

One set of RF hyperparameters was used for all ten diseases -> this implicitly assumes that:

- All diseases have similar noise levels

- All diseases have similar predictor relationships

- All models need equal depth and number of trees

- This is not true in real-world epidemiology.

After tuning:

- GridSearchCV provided individual best models, e.g.:

- diabetes -> deeper model with more estimators

- obesity -> shallower depth but higher learning of lifestyle variables

- COPD and stroke -> high-depth, high-tree models capturing nonlinear interactions

This improved:

- Generalization

- Stability of importance rankings

- Reduction of noise-driven importance

Thus, tuned RF importances are more trustworthy and provide disease-specific insights.

## 2. Key Predictors for Each Chronic Disease

Below is a concise summary of which variables had the strongest influence.

### Diabetes (DIABETES_CRUDEPREV)

Top predictors:

- selfcare_crudeprev

- phlth_crudeprev (poor physical health)

- lpa_crudeprev (lack of physical activity)

- disability_crudeprev

Why:
- Self-care disability and poor physical health directly reflect the functional decline associated with diabetes complications (neuropathy, mobility impairment). This is strongly supported by clinical literature.

### Obesity (OBESITY_CRUDEPREV)

Top predictors:

- csmoking_crudeprev

- lpa_crudeprev

- selfcare_crudeprev

Why:
- Smoking and obesity cluster as part of shared socioeconomic and behavioral patterns. Physical inactivity also plays a major role, aligning with CDC obesity research.

### High Blood Pressure (BPHIGH_CRUDEPREV)

Top predictors:

- checkup_crudeprev

- phlth_crudeprev

- disability_crudeprev

Why:
- Hypertension is asymptomatic and often detected only during routine checkups. Higher checkup rates -> higher diagnosed prevalence.

### Blood Pressure Medication (BPMED_CRUDEPREV)

Top predictors:

- checkup_crudeprev

- bphigh_crudeprev

- phlth_crudeprev

Why:
- Medication rates follow from diagnosed hypertension and healthcare-system engagement.

### Coronary Heart Disease (CHD_CRUDEPREV)

Top predictors:

- phlth_crudeprev

- lpa_crudeprev

- disability_crudeprev

Why:
- CHD leads to severe physical limitations, exercise intolerance, and long-term disability.

### Stroke (STROKE_CRUDEPREV)

Top predictors:

- phlth_crudeprev

- disability_crudeprev

- selfcare_crudeprev

Why:
- Stroke is a major cause of long-term disability. Poor physical health days capture post-stroke sequelae.

### COPD (COPD_CRUDEPREV)

Top predictors:

- phlth_crudeprev

- csmoking_crudeprev

- disability_crudeprev

Why:
- COPD causes chronic physical limitation and is strongly tied to smoking.

### Cancer (CANCER_CRUDEPREV)

Top predictors:

- colon_screen_crudeprev

- checkup_crudeprev

- phlth_crudeprev

Why:
- Cancer prevalence is driven heavily by screening efforts. Colon screening strongly influences detection.

### Arthritis (ARTHRITIS_CRUDEPREV)

Top predictors:

- checkup_crudeprev

- phlth_crudeprev

Why:
- Arthritis is a chronic condition requiring routine healthcare visits for management and medication.

### High Cholesterol (HIGHCHOL_CRUDEPREV)

Top predictors:

- cholscreen_crudeprev

- checkup_crudeprev

Why:
- Cholesterol must be measured through lab screening. Higher screening rates -> more diagnosed cases.

## 3. Consistency Between Random Forest, Correlation, and Medical Literature

A three-way validation was applied:

- (a) Correlation analysis matched RF importance for many diseases

        Diabetes -> self-care, poor physical health

        Stroke and COPD -> strong correlation with poor physical health

        CHD -> physical health + inactivity

- (b) RF identified nonlinear drivers not captured by correlation

    Example:

        binge_crudeprev had low correlation but meaningful RF importance -> indicates nonlinear or interaction-based patterns detected by the model.

- (c) Medical research strongly supports identified predictors

        Each top predictor corresponds to well-established disease mechanisms (screening -> detection, smoking -> COPD, disability -> CHD/stroke/diabetes).

## 4. Conclusion for RQ2

The tuned Random Forest models provide highly interpretable, medically valid feature importance rankings. The predictors influencing chronic diseases fall into three dominant categories:

- Functional health and disability

- Preventive care and screening access

- Behavioral risk factors

These patterns align with:

- Epidemiological theory

- Correlation structure

Thus, for all ten chronic disease outcomes, the most influential predictors were consistent, meaningful, and supported by both statistical evidence and clinical literature.

# RQ3: Composite Health Burden — Creating the Chronic Health Risk Index (CHRI)
How can multiple chronic disease outcomes be combined into a single CHRI score that summarizes community-level disease burden?

The primary goal of RQ3 is to design a single, interpretable, data-driven index that summarizes the chronic-disease burden of each U.S. census tract. Instead of using a manually weighted or arbitrary scoring system, we construct CHRI using a Neural Network (NN) that learns the optimal nonlinear combination of ten chronic disease prevalence values.

This provides a scientifically justified, statistically optimized, and interpretable method to combine multiple health outcomes into one continuous 0–100 risk index.

## 1. Motivation for Creating CHRI

Public-health datasets often contain many chronic conditions that are:

- Highly correlated in some cases (e.g., diabetes -> stroke -> CHD),

- Weakly or inconsistently correlated in others (e.g., obesity -> cancer),

- Nonlinearly related,

- Confounded by demographic and behavioral factors.

Our correlation heatmap clearly showed that disease relationships are non-uniform and nonlinear:

Strong clusters (e.g., diabetes–stroke, COPD–stroke, CHD–arthritis)

Weak or negative correlations for other pairs

Complex interaction patterns

Because of this heterogeneity:

A simple linear index would be misleading.

It would assume all diseases contribute equally and linearly — which contradicts epidemiological evidence.

Therefore, a neural-network–based index is more appropriate because it can learn:

nonlinearities,

unequal disease contributions,

interactions between conditions,

latent patterns of overall disease burden.

## 2. Method: Nonlinear Neural Network to Learn CHRI

We build a neural network with architecture:

10 disease inputs 
-> Dense(32, ReLU) 
-> Dense(16, ReLU) 
-> Dense(1, Linear Output)

Why this works:

Input layer (10 diseases) provides the full chronic-disease profile.

Hidden layers learn nonlinear interactions (e.g., diabetes + obesity jointly raising risk non-additively).

Output layer produces a single continuous score representing latent disease burden.

Training target:

Because CHRI does not exist in the dataset, we train the model to learn the mean of the 10 chronic conditions:

CHRI_target
=
1
10
∑
𝑖
=
1
10
Disease
𝑖
CHRI_target=
10
1
	​

i=1
∑
10
	​

Disease
i
	​


This gives the NN a baseline disease signal to learn, while enabling:

nonlinear weighting,

disease-interaction capture,

latent-risk extraction.

Convergence:

The NN trained efficiently with loss dropping from:

799 → 0.06 (MSE)

This indicates:

strong mapping learned,

stable convergence,

no overfitting (single dataset, no noise mismatch).

## 3. Scaling CHRI to a Usable 0–100 Score

The NN outputs raw continuous scores.
We normalize these into a 0–100 health risk index:

CHRI𝑠𝑐𝑎𝑙𝑒𝑑= 100×CHRI𝑟𝑎𝑤−
min
⁡
(
CHRI
𝑟
𝑎
𝑤
)
max
⁡
(
CHRI
𝑟
𝑎
𝑤
)
−
min
⁡
(
CHRI
𝑟
𝑎
𝑤
)
CHRI
scaled
	​

=100×
max(CHRI
raw
	​

)−min(CHRI
raw
	​

)
CHRI
raw
	​

−min(CHRI
raw
	​

)
	​


Properties of this transformation:

preserves relative ordering,

enhances interpretability,

makes the index comparable across tracts,

enables classification.

## 4. Creating CHRI Risk Tiers (Low -> High)

To classify communities into meaningful groups, we divide CHRI into quantile-based risk categories:

CHRI Percentile	Tier
Bottom 33%	Low Risk
Middle 33%	Medium Risk
Top 33%	High Risk

This creates balanced classes for downstream classification and ensures:

comparability across census tracts,

equitable representation of risk strata,

stable performance for ML classification models.

## 5. Validation: Does CHRI Represent True Disease Burden?

We validated CHRI through:

(a) Correlation with all 10 diseases

CHRI showed strong positive relationships with key conditions:

Highest correlations with:

- diabetes

- stroke

- COPD

- CHD

- arthritis

This matches clinical understanding:
communities with higher multimorbidity typically show high cardiometabolic and respiratory disease prevalence.

(b) Example tract comparison

When comparing a high-CHRI vs low-CHRI census tract:

- High-risk tracts showed consistently elevated prevalence across multiple diseases.

- Low-risk tracts had low values across the board.

- This demonstrates that CHRI successfully summarizes disease burden in a smooth, coherent, interpretable way.

(c) Epidemiological validity

CHRI aligns with the public health concept of multimorbidity, where diseases cluster due to shared:

- behavioral risks,

- socioeconomic stressors,

- genetic predispositions,

- environmental exposures.

CHRI therefore reflects real-world health vulnerability, not mathematical artifacts.

## 6. Why CHRI Works Better Than Linear Methods
Linear methods assume:

equal contributions of all diseases,

additivity,

constant relationships across tracts.

Our NN-based CHRI instead learns:

- nonlinear disease interactions,

- unequal weights,

- latent multimorbidity patterns.

This produces a richer, more realistic index of community chronic health risk.

## Conclusion for RQ3

The CHRI developed in this project is a robust, nonlinear, data-driven health risk index that:

- Combines ten chronic diseases into a single continuous metric

- Learns optimal nonlinear interactions via a neural network

- Produces interpretable 0–100 risk scores

- Accurately separates census tracts into meaningful low/medium/high risk categories

- Reflects real epidemiological patterns of multimorbidity

This fulfills the objective of creating a scientifically valid and operationally useful composite measure of chronic disease burden.

# RQ4: Can census tracts be reliably classified into Low, Medium, and High risk levels using the CHRI?

**Yes.**
The CHRI provides a reliable way to classify census tracts into Low, Medium, and High chronic-health-risk tiers because the score reflects meaningful patterns in the underlying disease data.

For example, tracts with low prevalence of severe conditions (such as stroke, COPD, and coronary heart disease), moderate metabolic indicators (like obesity and high cholesterol), and strong treatment adherence (e.g., high BPMED) consistently fall into the Low-risk tier.
This mirrors the behavior seen in the sample tract, where modest disease levels and good access to care naturally aligned with a low CHRI score.

Conversely, tracts with elevated cardiometabolic burden, high rates of unmanaged chronic illness, and lower preventive-care engagement rise into medium or high tiers.

This demonstrates that the CHRI captures structural differences in community health, allowing census tracts to be meaningfully and consistently grouped into risk levels.